# 02 - AMR Gene Matrix

Turn the AMRFinderPlus output for each genome into one table showing which
resistance genes each genome carries (1 = present, 0 = absent).

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.home() /"amr-project"
PROC = ROOT /"data" /"processed"
AMR_DIR = ROOT / "data" / "amr"
files = sorted(AMR_DIR.glob("*.tsv"))
print(len(files), "annotation files found")
print([f.name for f in files[:3]])

2440 annotation files found
['573.12771.tsv', '573.12772.tsv', '573.12773.tsv']


In [2]:
example=pd.read_csv(files[0], sep="\t")
print(example.shape)
example[["Element symbol", "Type", "Subtype", "Class"]].head(10)

(29, 22)


,Element symbol,Type,Subtype,Class
0,emrD,AMR,AMR,EFFLUX
1,fieF,STRESS,METAL,NaN
2,fosA,AMR,AMR,FOSFOMYCIN
3,blaSHV-1,AMR,AMR,BETA-LACTAM
4,ybtQ,VIRULENCE,VIRULENCE,NaN
5,ybtP,VIRULENCE,VIRULENCE,NaN
6,ompK36_K231SfsTer16,AMR,POINT_DISRUPT,BETA-LACTAM
7,oqxA11,AMR,AMR,PHENICOL/QUINOLONE
8,oqxB19,AMR,AMR,PHENICOL/QUINOLONE
9,dfrA1,AMR,AMR,TRIMETHOPRIM


In [3]:
frames = []
for f in files:
    df = pd.read_csv(f, sep="\t", dtype=str)
    df["genome_id"] = f.stem
    frames.append(df)

amr = pd.concat(frames, ignore_index=True)
print(amr.shape)
amr["Type"].value_counts()

(86661, 23)


Type
STRESS       45174
AMR          39264
VIRULENCE     2223
Name: count, dtype: int64

In [4]:
hits = amr[amr["Type"] == "AMR"]

matrix = (hits.assign(present=1)
              .pivot_table(index="genome_id", columns="Element symbol",
                           values="present", aggfunc="max", fill_value=0))

# keep genomes that had zero AMR hits (as all-zero rows)
all_ids = [f.stem for f in files]
matrix = matrix.reindex(all_ids, fill_value=0).astype(int)

print(matrix.shape)
matrix.iloc[:5, :8]

(2440, 563)


Element symbol,aac(3)-IId,aac(3)-IIe,aac(3)-IIg,aac(3)-IVa,aac(3)-Ib,aac(3)-VIa,aac(6'),aac(6')-33
genome_id,,,,,,,,
573.12771,0,0,0,0,0,0,0,0
573.12772,0,0,0,1,0,0,0,0
573.12773,0,0,0,0,0,0,0,0
573.12777,0,0,0,0,0,0,0,1
573.12778,0,0,0,1,0,0,0,0


In [5]:
labels = pd.read_csv(PROC / "labels_wide.csv", index_col=0, dtype={"genome_id": str})
labels.index = labels.index.astype(str)

print("Matrix genomes missing from labels:", len(set(matrix.index) - set(labels.index)))
print("Labelled genomes missing from matrix:", len(set(labels.index) & set(all_ids) - set(matrix.index)))

print("\nResistance genes per genome:")
print(matrix.sum(axis=1).describe())

print("\nMost common genes/mutations:")
matrix.sum().sort_values(ascending=False).head(15)

Matrix genomes missing from labels: 0
Labelled genomes missing from matrix: 0

Resistance genes per genome:
count    2440.000000
mean       15.940164
std         6.331464
min         0.000000
25%        12.000000
50%        17.000000
75%        19.000000
max        38.000000
dtype: float64

Most common genes/mutations:


Element symbol
emrD           2435
fosA           2368
oqxA           2252
parC_S80I      1492
blaCTX-M-15    1361
gyrA_S83I      1294
oqxB           1160
sul1            974
blaTEM-1        922
dfrA14          865
sul2            832
oqxB19          817
gyrA_D87N       812
aph(3'')-Ib     812
aph(6)-Id       808
dtype: int64

In [6]:
matrix.to_csv(PROC / "gene_matrix.csv")
print("Saved", matrix.shape, "to", PROC / "gene_matrix.csv")

Saved (2440, 563) to /home/homepcmike_ambasa_k1/amr-project/data/processed/gene_matrix.csv


In [8]:
zero = matrix.index[matrix.sum(axis=1) == 0]
print(len(zero), "genomes with no AMR hits")
print(labels.loc[labels.index.isin(zero)])

print("\nGenomes without emrD:", int((matrix["emrD"] == 0).sum()))

1 genomes with no AMR hits
           ceftriaxone  ciprofloxacin  gentamicin
genome_id                                        
573.46086          0.0            0.0         0.0

Genomes without emrD: 5


In [9]:
no_emrd = matrix.index[matrix["emrD"] == 0]

rows = []
for g in no_emrd:
    fna = ROOT / "data" / "genomes" / f"{g}.fna"
    rows.append({
        "genome_id": g,
        "AMR_hits": int(matrix.loc[g].sum()),
        "genome_size_MB": round(fna.stat().st_size / 1e6, 2),
        "ciprofloxacin": labels.loc[g, "ciprofloxacin"],
        "ceftriaxone": labels.loc[g, "ceftriaxone"],
        "gentamicin": labels.loc[g, "gentamicin"],
    })
pd.DataFrame(rows)

,genome_id,AMR_hits,genome_size_MB,ciprofloxacin,ceftriaxone,gentamicin
0,573.13049,16,5.59,1.0,1.0,1.0
1,573.13732,3,2.80,0.0,1.0,0.0
2,573.31059,8,2.04,1.0,1.0,1.0
3,573.46086,0,0.02,0.0,0.0,0.0
4,573.46111,2,4.85,0.0,0.0,0.0


In [10]:
sizes = pd.Series(
    {g: (ROOT / "data" / "genomes" / f"{g}.fna").stat().st_size / 1e6
     for g in matrix.index}
)

print(sizes.describe().round(2))

odd = sizes[(sizes < 4.5) | (sizes > 6.5)].sort_values()
print("\nGenomes outside 4.5-6.5 MB:", len(odd))
odd.round(2)

count    2440.00
mean        5.66
std         0.38
min         0.02
25%         5.53
50%         5.62
75%         5.72
max        11.52
dtype: float64

Genomes outside 4.5-6.5 MB: 29


573.46086     0.02
573.31059     2.04
573.13732     2.80
573.14412     3.97
573.7029      6.54
573.12886     6.55
573.7028      6.61
573.13249     6.63
573.15469     6.73
573.13885     6.81
573.14330     6.91
573.15551     6.91
573.13407     7.08
573.13968     7.15
573.13505     7.22
573.13622     7.27
573.12957     7.99
573.14450     8.22
573.13946     8.39
573.12987     8.75
573.15437     8.91
573.13822     8.96
573.13345     9.14
573.13353     9.15
573.15492     9.66
573.15570    10.21
573.15450    11.08
573.46106    11.31
573.31080    11.52
dtype: float64

In [11]:
removed = odd.index.tolist()
keep = matrix.index.difference(removed)

matrix_clean = matrix.loc[keep]
labels_clean = labels.loc[matrix_clean.index]

# drop genes that are now absent from every remaining genome
matrix_clean = matrix_clean.loc[:, matrix_clean.sum() > 0]

pd.Series(removed, name="genome_id").to_csv(PROC / "removed_genomes.csv", index=False)
matrix_clean.to_csv(PROC / "gene_matrix_clean.csv")
labels_clean.to_csv(PROC / "labels_clean.csv")

print("Removed:", len(removed), "genomes")
print("Clean matrix:", matrix_clean.shape)
print("Clean labels:", labels_clean.shape)

summary = (labels_clean.apply(lambda s: s.value_counts())
                       .rename(index={0.0: "Susceptible", 1.0: "Resistant"}).T)
print(summary)

Removed: 29 genomes
Clean matrix: (2411, 543)
Clean labels: (2411, 3)
               Susceptible  Resistant
ceftriaxone            410       2001
ciprofloxacin          570       1841
gentamicin            1408       1003


## Quality control and summary

- AMRFinderPlus output from 2,440 genomes was combined into a presence/absence
  matrix of 563 AMR genes and resistance mutations (STRESS and VIRULENCE rows excluded).
- Every labelled genome is in the matrix and vice versa (0 mismatches).
- Genome sizes were checked: the middle 50% fall between 5.53 and 5.72 MB
  (median 5.62 MB), as expected for *K. pneumoniae*.
- **29 genomes (1.2%) were removed** because their size fell outside 4.5-6.5 MB.
  Four were too small (0.02-3.97 MB), indicating incomplete assemblies, and 25 were
  too large (6.54-11.52 MB), suggesting contamination or duplicated assemblies.
  Their IDs are saved in `data/processed/removed_genomes.csv`.
- Genomes lacking emrD that passed the size check (573.13049, 573.46111) were kept.
- Clean outputs for modelling: `gene_matrix_clean.csv` and `labels_clean.csv`.
  The original uncleaned matrix is also kept as `gene_matrix.csv`.